# Calculation Area Model Execution

This notebook executes the completed model training run from the standalone notebook via `%run`, then propagates the exact methodology across a new geometric region.

In [57]:
%run ../svm/forest_classification_best.ipynb


Forest points: 250
Non-forest points: 250
[[71, 0], [2, 79]]
Accuracy: 0.9868421052631579
Kappa: 0.9736156917201874


In [58]:
import math

# Variables campus, dataset, mask_s2_clouds, and bands are imported globally from the %run execution.
area = campus.area()
print("Campus Area (m²):", area.getInfo())
print("Campus Area (km²):", area.divide(1e6).getInfo())

targeted_area_sqm = float(area.getInfo())*2
point = {"type": "Point", "coordinates": [79.88361463362526,23.142324404777362]}
centre = ee.Geometry(point)
circle_radius = math.sqrt(targeted_area_sqm / math.pi)
roi = centre.buffer(circle_radius)


Campus Area (m²): 1995406.3843647253
Campus Area (km²): 1.9954063843645904


In [59]:
dataset_new = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate('2025-11-01', '2025-12-31')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(mask_s2_clouds))
new_image = dataset_new.median().clip(roi)
ndvi_new = new_image.normalizedDifference(['B8', 'B4']).rename('NDVI')
new_image = new_image.addBands(ndvi_new)


In [60]:
new_classified = new_image.select(bands).classify(classifier)
new_smooth_classification = new_classified.focalMode(1)


In [61]:
Map2 = geemap.Map()
Map2.centerObject(roi, 16)
Map2.addLayer(new_image, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Test Area RGB')
Map2.addLayer(new_smooth_classification, {'min': 0, 'max': 1, 'palette': ['lightgray', 'darkgreen']}, 'Test Area Forest Map')
Map2


Map(center=[23.142325561034358, 79.8836146684978], controls=(WidgetControl(options=['position', 'transparent_b…

In [62]:
test_forest_points = ee.FeatureCollection('users/cosypix/test_forest_points')
test_non_forest_points = ee.FeatureCollection('users/cosypix/test_non_forest_points')
test_points = test_forest_points.merge(test_non_forest_points)
def _count(fc):
    try:
        return int(fc.size().getInfo())
    except Exception as e:
        return None
print("Forest points:", _count(test_forest_points))
print("Non-forest points:", _count(test_non_forest_points))


Forest points: 240
Non-forest points: 240


In [63]:
validation_data = new_image.select(bands).sampleRegions(
    collection=test_points, properties=['label'], scale=10)
validation_data = validation_data.filter(ee.Filter.notNull(bands + ['label']))
validated = validation_data.classify(classifier)
confusion_matrix = validated.errorMatrix(actual='label', predicted='classification')
print("Confusion Matrix: ", confusion_matrix.getInfo())
print("Test Area Accuracy: ", confusion_matrix.accuracy().getInfo())
print("Test Area Kappa: ", confusion_matrix.kappa().getInfo())


Confusion Matrix:  [[229, 11], [62, 178]]
Test Area Accuracy:  0.8479166666666667
Test Area Kappa:  0.6958333333333333
